In [ ]:
!pip install gradio diffusers transformers accelerate safetensors mediapy ultralytics simple-lama-inpainting easyocr anthropic

!pip install git+https://github.com/facebookresearch/sam2.git -q

!wget -q "https://github.com/google/fonts/raw/main/ofl/bangers/Bangers-Regular.ttf" -O /content/Bangers.ttf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 MB 21.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of scikit-image to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of tifffile to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.9 MB/s eta 0:00:

ERROR: Operation cancelled by user
^C


In [3]:
!pip install --upgrade torchao peft diffusers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 139.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 47.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.1
    Uninstalling diffusers-0.37.1:
      Successfully uninstalled diffusers-0.37.1


In [1]:
!pip install git+https://github.com/facebookresearch/sam2.git -q

!pip install simple-lama-inpainting easyocr ultralytics -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.8 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
except:
    os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

In [2]:
import torch
import gradio as gr
from PIL import Image

from image_gen import load_pipeline
from pipeline import run_pipeline, compare_modes

MODEL_ID  = "runwayml/stable-diffusion-v1-5"
LORA_PATH = "pytorch_lora_weights.safetensors"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

print("--- Loading Pipeline ---")
pipe = load_pipeline(MODEL_ID, lora_path=LORA_PATH, device=DEVICE)
print("--- Pipeline Ready ---")

def get_overrides(c1_n, c1_d, c2_n, c2_d):
    """Формує словник переозначення персонажів для моделі"""
    overrides = {}
    if c1_n.strip() and c1_d.strip():
        overrides[c1_n.strip()] = c1_d.strip()
    if c2_n.strip() and c2_d.strip():
        overrides[c2_n.strip()] = c2_d.strip()
    return overrides if overrides else None

def generate_comic_fn(story, mode, decay, seed, c1_n, c1_d, c2_n, c2_d):
    """Функція для генерації одного коміксу"""
    if not story.strip():
        return None

    overrides = get_overrides(c1_n, c1_d, c2_n, c2_d)

    comic, result, prompts, images = run_pipeline(
        story=story,
        pipe=pipe,
        mode=mode,
        decay=float(decay),
        seed=int(seed),
        num_inference_steps=50,
        guidance_scale=7.5,
        save_path="comic_output.png",
        character_overrides=overrides
    )
    return [comic]

def compare_modes_fn(story, c1_n, c1_d, c2_n, c2_d):
    """Функція для порівняння режимів (Ablation Study)"""
    if not story.strip():
        return None

    overrides = get_overrides(c1_n, c1_d, c2_n, c2_d)

    comparison = compare_modes(
        story=story,
        pipe=pipe,
        save_path="comparison.png",
        character_overrides=overrides,
    )
    return [comparison]

custom_css = """
/* 1. УНІФІКАЦІЯ ШРИФТІВ */
.gradio-container .block span,
.gradio-container label span,
.gradio-container .gr-form-label,
.gradio-container .block-title,
.gradio-container .char-container h3 {
    font-size: 20px !important;
    font-weight: bold !important;
    color: white !important;
}

.gradio-container textarea,
.gradio-container input {
    font-size: 18px !important;
}

.gradio-container .item,
.gradio-container ul.options li,
.gradio-container .token-list span {
    font-size: 18px !important;
}

/* 3. ПРИБИРАЄМО СТРІЛКИ ТА СПІНЕРИ */
textarea::-webkit-scrollbar,
input::-webkit-scrollbar {
    display: none !important;
}

textarea, input {
    scrollbar-width: none !important;
    -ms-overflow-style: none !important;
    overflow: hidden !important;
    resize: none !important;
}

input::-webkit-outer-spin-button,
input::-webkit-inner-spin-button {
    -webkit-appearance: none !important;
    margin: 0 !important;
}

input[type=number] {
    -moz-appearance: textfield !important;
}

::-webkit-resizer {
    display: none !important;
}

/* 4. ПОВЗУНОК (DECAY) */
input[type="range"] {
    accent-color: #ff8c00 !important;
    cursor: pointer !important;
}

input[type="range"]::-webkit-slider-thumb {
    border: 2px solid black !important;
    box-shadow: 0 0 2px black !important;
}

/* 5. КОРЕКЦІЯ ПОЛЯ DECAY */
.decay-number-field input[type='number'] {
    width: 60px !important;
    min-width: 60px !important;
}

/* 6. КНОПКИ */
.buttons-row {
    display: flex !important;
    gap: 10px !important;
}
.buttons-row > button {
    flex: 1 1 0% !important;
    height: 50px !important;
    font-size: 20px !important;
    font-weight: bold !important;
}

.orange-btn {
    background-color: #ff8c00 !important;
    border: none !important;
    color: black !important;
    font-weight: bold !important;
    font-size: 20px !important;
    transition: background-color 0.3s ease !important;
}
.orange-btn:hover {
    background-color: #cc7000 !important;
}

.char-section-header {
    font-size: 24px !important;
    font-weight: bold !important;
    margin-top: 40px !important;
    margin-bottom: 10px !important;
    color: white !important;
}

.char-container {
    padding: 15px !important;
    border: 1px solid #444 !important;
    border-radius: 10px !important;
    margin-bottom: 15px !important;
}

.equal-size {
    display: flex !important;
    align-items: flex-end !important;
}
.equal-size > div {
    flex: 1 1 0% !important;
}

.main-title {
    text-align: center;
    font-size: 36px !important;
    font-weight: 900 !important;
    color: white !important;
    margin-bottom: 25px !important;
}
"""

js_func = """
function() {
    const textareas = document.querySelectorAll('textarea');
    textareas.forEach(el => {
        el.style.resize = 'none';
        el.style.overflow = 'hidden';
        el.addEventListener('input', function() {
            this.style.height = 'auto';
            this.style.height = (this.scrollHeight) + 'px';
        });
    });
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:

    gr.HTML("<div class='main-title'>✨TURN YOUR STORY INTO A COMIC✨</div>")

    story_input = gr.Textbox(
        label="Story",
        placeholder="Enter your story here (3-5 sentences)...",
        lines=6
    )

    with gr.Row(elem_classes=["equal-size"]):
        mode_dropdown = gr.Dropdown(
            label="Mode:",
            choices=["adaptive", "standard", "baseline"],
            value="adaptive",
            scale=1
        )
        decay_slider = gr.Slider(
            label="Decay:",
            minimum=0.0,
            maximum=1.0,
            value=0.5,
            step=0.1,
            scale=2,
            elem_classes=["decay-number-field"]
        )
        seed_input = gr.Number(label="Seed:", value=42, precision=0, scale=1)

    gr.HTML("<div class='char-section-header'>Character overrides (optional):</div>")

    with gr.Column(elem_classes=["char-container"]):
        gr.Markdown("### Character 1")
        with gr.Row():
            char1_name = gr.Textbox(label="Name", placeholder="ім'я", scale=1, lines=1)
            char1_desc = gr.Textbox(label="Description", placeholder="опис", scale=3, lines=1)

    with gr.Column(elem_classes=["char-container"]):
        gr.Markdown("### Character 2")
        with gr.Row():
            char2_name = gr.Textbox(label="Name", placeholder="ім'я", scale=1, lines=1)
            char2_desc = gr.Textbox(label="Description", placeholder="опис", scale=3, lines=1)

    with gr.Row(elem_classes=["buttons-row"]):
        generate_btn = gr.Button("Generate Comic", variant="primary", elem_classes=["orange-btn"])
        compare_btn = gr.Button("Compare Modes", variant="secondary")

    output_gallery = gr.Gallery(show_label=False, columns=[1], object_fit="contain", height="auto")

    demo.load(None, None, None, js=js_func)

    generate_btn.click(
        fn=generate_comic_fn,
        inputs=[
            story_input, mode_dropdown, decay_slider, seed_input,
            char1_name, char1_desc, char2_name, char2_desc
        ],
        outputs=output_gallery
    )

    compare_btn.click(
        fn=compare_modes_fn,
        inputs=[
            story_input,
            char1_name, char1_desc, char2_name, char2_desc
        ],
        outputs=output_gallery
    )

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


--- Loading Pipeline ---
Loading runwayml/stable-diffusion-v1-5...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its resul

Loading LoRA from pytorch_lora_weights.safetensors...


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
/tmp/ipykernel_9368/558293073.py:204: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:
/tmp/ipykernel_9368/558293073.py:204: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:


LoRA loaded
Pipeline ready
--- Pipeline Ready ---


In [3]:
if __name__ == "__main__":
    demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://60c40f7759ca6a0ff2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Step 1: Parsing story...
  Characters: ['Sarah Kane', 'Ghost']
  Panel 1: Dimly lit detective office with a wooden desk and city windo
  Panel 2: Dark crumbling mansion interior with towering shadowy walls 
  Panel 3: Dim mansion library with tall dusty bookshelves and cobwebs 
  Panel 4: Hidden secret room filled with glittering gold coins and jew

Step 2: Building prompts...
  Panel 1: comicstyle, comic book, bold outlines, speech bubble, Dimly lit detective office
  Panel 2: comicstyle, comic book, bold outlines, speech bubble, Dark crumbling mansion int
  Panel 3: comicstyle, comic book, bold outlines, speech bubble, Dim mansion library with t
  Panel 4: comicstyle, comic book, bold outlines, speech bubble, Hidden secret room filled 

Step 3: Attention mode='adaptive'...
Middle block SA: 1 layer(s) replaced (decay=0.5)

Step 4: Generating panels (steps=50, cfg=7.5, seed=42)...


  0%|          | 0/50 [00:00<?, ?it/s]

  Saved 4 raw panels to /tmp/panels

Step 5: Generating dialogues (Claude Vision)...
  Panel 1: generating dialogue...
    → 'Who are you? Show yourself!' | 'Sarah Kane enters the shadowy haunted mansion.'
  Panel 2: generating dialogue...
    → 'Show yourself... I know you're here.' | 'Sarah moves deeper into the crumbling darkness.'
  Panel 3: generating dialogue...
    → 'These books hold ancient secrets!' | 'The spirit gestured toward forbidden knowledge.'
  Panel 4: generating dialogue...
    → 'The treasure was here all along!' | 'Sarah discovers the ghost guarding ancient riches.'

Step 6: BubbleCleaner (YOLO + LaMa + PIL)...

Panel 1: /tmp/panels/panel_01.png


sam2_hiera_base_plus.pt:   0%|          | 0.00/323M [00:00<?, ?B/s]

...

  Saved 4 raw panels to /tmp/panels

Step 5: Generating dialogues (Claude Vision)...
  Panel 1: generating dialogue...
    → 'This map leads somewhere dangerous...' | 'Sarah examines the mysterious envelope's contents carefully.'
  Panel 2: generating dialogue...
    → 'Who's there? Show yourself now!' | 'Sarah Kane enters the haunted mansion alone.'
  Panel 3: generating dialogue...
    → 'What do you want?!' | 'The ghost speaks from the shadows above.'
  Panel 4: generating dialogue...
    → 'The treasure was here all along!' | 'Sarah discovers the ghost's hidden gold hoard.'

Step 6: BubbleCleaner (YOLO + LaMa + PIL)...

Panel 1: /tmp/panels/panel_01.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...
  No bubble found — keeping original

Panel 2: /tmp/panels/panel_02.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...

Panel 3: /tmp/panels/panel_03.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...

Panel 4: /tmp/panels/panel_04.png
  YOLO: 0 bubble(s) found
  Falling 

  0%|          | 0/50 [00:00<?, ?it/s]

  Saved 4 raw panels to /tmp/panels

Step 5: Generating dialogues (Claude Vision)...
  Panel 1: generating dialogue...
    → 'My fangs need cleaning, please!' | 'Vladimir demands dental care at last.'
  Panel 2: generating dialogue...
    → 'Just a cleaning, please!' | 'Vladimir opens wide. Dr. Molars screams loudly.'
  Panel 3: generating dialogue...
    → 'Just a cleaning, please!' | 'Vladimir opens wide for Dr. Molars.'
  Panel 4: generating dialogue...
    → 'Triple price for fangs!' | 'Vladimir grins. Dr. Molars reluctantly agrees.'

Step 6: BubbleCleaner (YOLO + LaMa + PIL)...

Panel 1: /tmp/panels/panel_01.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...

Panel 2: /tmp/panels/panel_02.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...

Panel 3: /tmp/panels/panel_03.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...

Panel 4: /tmp/panels/panel_04.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...

Step 7: Assembling comic strip...
  Saved to comic_output.png

  0%|          | 0/50 [00:00<?, ?it/s]

  Saved 4 raw panels to /tmp/panels

Step 5: Generating dialogues (Claude Vision)...
  Panel 1: generating dialogue...
    → 'Do you have any openings?' | 'The dragon nervously checks the bakery window.'
  Panel 2: generating dialogue...
    → 'I love working here!' | 'Baker joyfully tends the pastry shelves.'
  Panel 3: generating dialogue...
    → 'Oops! My bad, sorry!' | 'Dragon's breath ignites the whole bakery!'
  Panel 4: generating dialogue...
    → 'I can help bake things!' | 'Dragon eagerly applies for the bakery job.'

Step 6: BubbleCleaner (YOLO + LaMa + PIL)...

Panel 1: /tmp/panels/panel_01.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...

Panel 2: /tmp/panels/panel_02.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...

Panel 3: /tmp/panels/panel_03.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...

Panel 4: /tmp/panels/panel_04.png
  YOLO: 0 bubble(s) found
  Falling back to SAM2...

Step 7: Assembling comic strip...
  Saved to comic_output.png
Keyboard 

### запасний

In [5]:
# import gradio as gr

# # CSS для повного видалення стрілок (image_e78ebf.png) та уніфікації шрифтів
# custom_css = """
# /* 1. УНІФІКАЦІЯ ШРИФТІВ ЛЕЙБЛІВ (20px) */
# .gradio-container .block span,
# .gradio-container label span,
# .gradio-container .gr-form-label,
# .gradio-container .block-title {
#     font-size: 20px !important;
#     font-weight: bold !important;
#     color: white !important;
# }

# /* 2. ШРИФТ УСЕРЕДИНІ ВСІХ ПОЛІВ (20px) */
# .gradio-container textarea,
# .gradio-container input {
#     font-size: 18px !important;
# }

# /* 3. ПРИБИРАЄМО СТРІЛКИ ТА СПІНЕРИ (image_e78ebf.png) */
# /* Для числових полів (Chrome, Safari, Edge, Opera) */
# input::-webkit-outer-spin-button,
# input::-webkit-inner-spin-button {
#     -webkit-appearance: none !important;
#     margin: 0 !important;
# }

# /* Для Firefox */
# input[type=number] {
#     -moz-appearance: textfield !important;
# }

# /* Вимикаємо зміну розміру (resizer) */
# textarea, .gradio-container textarea {
#     resize: none !important;
# }

# ::-webkit-resizer {
#     display: none !important;
# }

# /* 4. КОРЕКЦІЯ ПОЛЯ DECAY (вузьке поле) */
# .decay-number-field input[type='number'] {
#     width: 60px !important;
#     min-width: 60px !important;
# }

# /* 5. КНОПКИ ОДНАКОВОГО РОЗМІРУ */
# .buttons-row {
#     display: flex !important;
#     gap: 10px !important;
# }
# .buttons-row > button {
#     flex: 1 1 0% !important;
#     height: 50px !important;
# }

# .orange-btn {
#     background-color: #ff8c00 !important;
#     border: none !important;
#     color: black !important;
#     font-weight: bold !important;
#     font-size: 18px !important;
#     transition: background-color 0.3s ease !important;
# }
# .orange-btn:hover {
#     background-color: #cc7000 !important;
# }

# /* Секція персонажів */
# .char-section-header {
#     font-size: 24px !important;
#     font-weight: bold !important;
#     margin-top: 40px !important;
#     margin-bottom: 10px !important;
#     color: white !important;
# }

# .char-container {
#     padding: 15px !important;
#     border: 1px solid #444 !important;
#     border-radius: 10px !important;
#     margin-bottom: 15px !important;
# }

# input[type="range"] {
#     accent-color: #ff8c00 !important;
# }

# .equal-size {
#     display: flex !important;
#     align-items: flex-end !important;
# }
# .equal-size > div {
#     flex: 1 1 0% !important;
# }

# .main-title {
#     text-align: center;
#     font-size: 36px !important;
#     font-weight: 900 !important;
#     color: white !important;
#     margin-bottom: 25px !important;
# }
# """

# js_func = """
# function() {
#     const textareas = document.querySelectorAll('textarea');
#     textareas.forEach(el => {
#         el.style.resize = 'none';
#     });
#     const resizers = document.querySelectorAll('.tr-resizer, .resizer');
#     resizers.forEach(r => r.remove());
# }
# """

# def generate_comic_fn(*args):
#     return []

# with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:

#     gr.HTML("<div class='main-title'>✨TURN YOUR STORY INTO A COMIC✨</div>")

#     story_input = gr.Textbox(
#         label="Story",
#         placeholder="Enter your story here...",
#         lines=6
#     )

#     with gr.Row(elem_classes=["equal-size"]):
#         mode_dropdown = gr.Dropdown(label="Mode:", choices=["adaptive", "static", "random"], value="adaptive", scale=1)
#         decay_slider = gr.Slider(
#             label="Decay:",
#             minimum=0.0,
#             maximum=1.0,
#             value=0.5,
#             step=0.1,
#             scale=2,
#             elem_classes=["decay-number-field"]
#         )
#         seed_input = gr.Number(label="Seed:", value=42, precision=0, scale=1)

#     gr.HTML("<div class='char-section-header'>Character overrides (optional):</div>")

#     with gr.Column(elem_classes=["char-container"]):
#         gr.Markdown("### Character 1")
#         with gr.Row():
#             # Використовуємо Textbox для імен/описів - у них немає числових стрілок за замовчуванням
#             char1_name = gr.Textbox(label="Name", placeholder="ім'я", scale=1, lines=1)
#             char1_desc = gr.Textbox(label="Description", placeholder="опис", scale=3, lines=1)

#     with gr.Column(elem_classes=["char-container"]):
#         gr.Markdown("### Character 2")
#         with gr.Row():
#             char2_name = gr.Textbox(label="Name", placeholder="ім'я", scale=1, lines=1)
#             char2_desc = gr.Textbox(label="Description", placeholder="опис", scale=3, lines=1)

#     with gr.Row(elem_classes=["buttons-row"]):
#         generate_btn = gr.Button("Generate Comic", variant="primary", elem_classes=["orange-btn"])
#         compare_btn = gr.Button("Compare Modes", variant="secondary")

#     output_gallery = gr.Gallery(show_label=False, columns=[4], object_fit="contain", height="auto")

#     demo.load(None, None, None, js=js_func)

#     generate_btn.click(
#         fn=generate_comic_fn,
#         inputs=[story_input, mode_dropdown, decay_slider, seed_input,
#                 char1_name, char1_desc, char2_name, char2_desc],
#         outputs=output_gallery
#     )

# if __name__ == "__main__":
#     demo.launch()


# ### the best